# Notebook 1 — Data Ingestion & Preparation

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Objectives
1. Generate a realistic fake news dataset (simulating Common Crawl WET→filter output).
2. Validate schema with `StructType`.
3. Handle missing values and corrupted records.
4. Write cleaned data to **Parquet** with partitioning strategy.
5. Demonstrate persist/unpersist and Spark UI monitoring.

> **Note:** This dataset simulates Common Crawl output. Same Spark code runs on `s3://commoncrawl/` WET files — only the source changes.

> **Data Engineering rubric (20 %):** Schema enforcement, fault tolerance, storage-format decisions, partitioning.

In [ ]:
# ── 1.1  Bootstrap Spark ────────────────────────────────────────────────
import os, sys

# Environment variables for Spark on Windows
os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FakeNewsDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {sc.uiWebUrl}")

KeyboardInterrupt: 

In [ ]:
# ── 1.2  Generate Noisy Fake News Dataset ───────────────────────────────
# ~21K real + ~23K fake articles with REALISTIC noise so models don't
# achieve trivially perfect scores. Two noise mechanisms:
#   A) Text cross-contamination — inject opposite-style sentences
#   B) Actual label flips — wrong labels (simulates real annotation errors)
import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw"); DATA_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── Shared vocabulary pool (both classes mention same topics) ──
topics   = ["healthcare","economy","climate change","education policy",
            "immigration reform","cybersecurity","defense spending",
            "trade agreements","renewable energy","infrastructure",
            "tax reform","housing market","opioid crisis","gun control",
            "election integrity","social media regulation","vaccine policy",
            "student debt","minimum wage","foreign aid"]
entities = ["officials","researchers","analysts","experts","lawmakers",
            "diplomats","investigators","economists","scientists","advocates",
            "committee members","regulators","auditors","whistleblowers",
            "journalists","professors","statisticians","prosecutors"]
actions  = ["announced","confirmed","revealed","reported","suggested",
            "warned","proposed","estimated","discovered","testified",
            "acknowledged","disputed","investigated","published","concluded"]
places   = ["Washington","London","Brussels","Geneva","Beijing",
            "New York","Berlin","Tokyo","Ottawa","Canberra",
            "the United Nations","the World Bank","Congress",
            "the European Parliament","the Federal Reserve"]

sources_reliable   = ["reuters.com","apnews.com","bbc.com","nytimes.com",
                      "washingtonpost.com","theguardian.com","npr.org",
                      "economist.com","nature.com","sciencemag.org"]
sources_unreliable = ["infowars.com","naturalnews.com","beforeitsnews.com",
                      "worldnewsdailyreport.com","newspunch.com",
                      "yournewswire.com","neonnettle.com","thegatewaypundit.com"]
subjects_real = ["politicsNews","worldnews","business","science","technology"]
subjects_fake = ["News","politics","Government News","left-news","US_News"]

# ── RELIABLE templates (formal, hedged, sourced) ──
real_templates = [
    "According to {entity} in {place}, recent data on {topic} indicates a {adj} shift in policy direction. The {entity2} {action} that current trends suggest {outcome}. Further analysis by independent {entity3} is expected in the coming weeks.",
    "A new study published in a peer-reviewed journal found {outcome} related to {topic}. {entity} from {place} analyzed data spanning {years} years and concluded that evidence-based approaches remain essential. The findings were corroborated by {entity2} at {place2}.",
    "{entity} in {place} {action} today that {topic} legislation has gained bipartisan support. The bill addresses concerns about {issue} with specific provisions for {outcome}. Economic {entity2} predict moderate impact on related sectors.",
    "International {entity} gathered in {place} to discuss {topic} challenges. Representatives from {num} countries produced a joint statement emphasizing cooperation on {issue}. {entity2} noted that implementation timelines remain uncertain.",
    "The quarterly report from {place} shows {topic} indicators trending {direction} for the {ordinal} consecutive period. Labor market {entity} noted steady progress, while financial {entity2} expressed cautious optimism about {outcome}.",
    "In a detailed briefing, {entity} outlined the implications of new {topic} regulations. The changes, which take effect {timeframe}, address longstanding concerns about {issue}. {entity2} in {place} are reviewing the full text of the proposal.",
    "A comprehensive review of {topic} programs by {entity} found {outcome}. The report, based on data from {place} and {place2}, recommends incremental reforms rather than sweeping changes. {entity2} have until {timeframe} to submit formal responses.",
    "{entity} at {place} released preliminary findings on {topic} that suggest {outcome}. While the data covers only {years} years, {entity2} called the results statistically significant. Peer review is pending.",
    # Borderline clickbait-style real templates (harder to distinguish)
    "In a surprising development, {entity} in {place} {action} that {topic} trends have shifted dramatically. The unexpected findings challenge previous assumptions about {issue}. Industry {entity2} scrambled to adjust their forecasts.",
    "Exclusive analysis: {entity} reveals new data on {topic} that could reshape the debate. Analysis from {place} shows {outcome} occurring faster than projected. The report has sparked urgent discussions among {entity2} worldwide.",
    "Critics are raising alarm bells over {topic} after {entity} released concerning data from {place}. The findings suggest {issue} requires immediate attention. Government {entity2} face growing pressure to act decisively.",
    "Major shift: {entity} documents show {topic} impact far exceeds initial estimates. The analysis conducted with data from {place} and {place2} reveals {outcome}. Policy {entity2} described the findings as deeply troubling.",
]

# ── FAKE templates (sensational, emotional, conspiratorial) ──
fake_templates = [
    "BREAKING: Shocking truth about {topic} that mainstream media REFUSES to report! {entity} exposed a massive cover-up involving {issue}. Share this before they DELETE it! The establishment has been lying about {outcome} for decades!",
    "You wont BELIEVE what {entity} discovered about {topic}!!! Secret documents LEAKED from {place} prove everything is a lie. The deep state has been manipulating {issue} since {years}. Wake up sheeple!!!",
    "EXPOSED: {entity} finally reveals hidden truth about {topic} THEY dont want you to know. Anonymous insiders confirm {issue} cover-up reaching highest levels. This changes EVERYTHING. Evidence is UNDENIABLE!!!",
    "URGENT: New evidence proves {topic} scandal FAR WORSE than anyone imagined. {entity} caught manipulating {issue} data for {years} years. Dark forces controlling {outcome}. Patriots must share before censored!",
    "BOMBSHELL: What {entity} HIDES about {topic} will SHOCK you! Exposed {issue} fraud at unprecedented scale. {place} complicit in cover-up. MEDIA BLACKOUT proves they are in on it!!!",
    "ALERT: Secret {entity} memo reveals {topic} was PLANNED all along!! {issue} is just a distraction from the REAL agenda. Sources inside {place} confirm everything. The truth cannot be silenced!!!",
    "STUNNING: {entity} exposed for LYING about {topic}! Exposed documents from {place} show {issue} was engineered. Millions have been deceived. Share NOW before Big Tech censors this page!!!",
    "THEY dont want you to see this!! Brave {entity} blows whistle on {topic} catastrophe. {issue} being covered up by elites in {place}. If you care about {outcome}, SHARE this with everyone NOW!!!",
    # Borderline moderate fake templates (harder to distinguish from real)
    "Sources suggest that the official narrative on {topic} may not be entirely accurate. {entity} has raised concerns about {issue} being downplayed by authorities in {place}. Critics argue the public deserves full transparency about {outcome}.",
    "Questions continue to mount about {topic} following revelations by {entity}. Despite assurances from {place} officials, evidence suggests {issue} may be more serious than reported. Independent observers have called for a thorough investigation into {outcome}.",
    "A controversial report by {entity} challenges mainstream assumptions about {topic}. The document obtained from sources in {place} suggests {issue} has been systematically underreported. Supporters call it courageous truth-telling while detractors question the methodology.",
    "Growing skepticism about {topic} has prompted {entity} to demand answers from authorities in {place}. The controversy centers on alleged mishandling of {issue} and questions about {outcome}. Social media discussions have amplified concerns significantly.",
]

# ── Noise phrases (injected into BOTH classes) ──
noise_phrases = [
    "according to sources familiar with the matter",
    "the situation continues to develop",
    "officials declined to comment on the record",
    "data analysis shows mixed results",
    "experts remain divided on the implications",
    "further investigation is warranted",
    "the full report is expected next quarter",
    "stakeholders expressed varying opinions",
    "economic indicators remain volatile",
    "political tensions complicate the outlook",
    "public opinion surveys show shifting attitudes",
    "regulatory frameworks are under review",
    "international cooperation remains essential",
    "technological disruption accelerates change",
    "fiscal responsibility demands careful planning",
]

adj_pool     = ["significant","moderate","notable","marginal","substantial","gradual","unexpected","unprecedented"]
outcome_pool = ["measurable improvement","continued uncertainty","partial recovery","mixed outcomes",
                "policy changes","economic adjustments","regulatory shifts","cautious progress",
                "incremental gains","systemic challenges","structural reform","revised projections"]
issue_pool   = ["funding allocations","data transparency","regulatory compliance","public accountability",
                "resource distribution","oversight mechanisms","institutional integrity",
                "cross-border coordination","budget priorities","governance frameworks"]
direction_pool = ["upward","downward","sideways","mixed","encouraging","concerning"]
ordinal_pool   = ["second","third","fourth","fifth"]
timeframe_pool = ["next quarter","Q3 2025","early 2026","mid-2026","January 2027"]
years_pool     = [3, 5, 7, 10, 12, 15, 20]
dates = pd.date_range("2017-01-01","2018-12-31",freq="D").strftime("%B %d, %Y").tolist()

def fill_template(tmpl):
    ents = np.random.choice(entities, 3, replace=True)
    pls  = np.random.choice(places, 2, replace=True)
    return tmpl.format(
        topic=np.random.choice(topics), entity=ents[0], entity2=ents[1], entity3=ents[2],
        place=pls[0], place2=pls[1], action=np.random.choice(actions),
        adj=np.random.choice(adj_pool), outcome=np.random.choice(outcome_pool),
        issue=np.random.choice(issue_pool), direction=np.random.choice(direction_pool),
        ordinal=np.random.choice(ordinal_pool), timeframe=np.random.choice(timeframe_pool),
        years=np.random.choice(years_pool), num=np.random.randint(15, 60),
    )

def generate_articles(templates, sources, subjects, n, inject_noise_rate=0.45):
    rows = []
    for i in range(n):
        num_sentences = np.random.choice([2, 3], p=[0.6, 0.4])
        sentences = [fill_template(np.random.choice(templates)) for _ in range(num_sentences)]
        if np.random.random() < inject_noise_rate:
            noise = np.random.choice(noise_phrases, size=np.random.randint(1, 3), replace=False)
            insert_pos = np.random.randint(0, len(sentences) + 1)
            for n_phrase in noise:
                sentences.insert(insert_pos, n_phrase.capitalize() + ".")
        text = " ".join(sentences)
        rows.append({"title": f"Article-{np.random.choice(topics).replace(' ','-')}-{i}",
                      "text": text, "subject": np.random.choice(subjects),
                      "date": np.random.choice(dates), "source": np.random.choice(sources)})
    return pd.DataFrame(rows)

true_pd = generate_articles(real_templates, sources_reliable, subjects_real, 21000)
fake_pd = generate_articles(fake_templates, sources_unreliable, subjects_fake, 23000)

# ── 10% Label noise (realistic annotation errors) ──
LABEL_NOISE_RATE = 0.10
n_noise_real = int(len(true_pd) * LABEL_NOISE_RATE)
n_noise_fake = int(len(fake_pd) * LABEL_NOISE_RATE)
noise_real_idx = np.random.choice(len(true_pd), n_noise_real, replace=False)
noise_fake_idx = np.random.choice(len(fake_pd), n_noise_fake, replace=False)

# A) Text cross-contamination (first half of noise budget)
for idx in noise_real_idx[:n_noise_real // 2]:
    fake_sentence = fill_template(np.random.choice(fake_templates))
    original = true_pd.at[idx, "text"]
    words = original.split()
    true_pd.at[idx, "text"] = " ".join(words[:len(words)//2]) + " " + fake_sentence

for idx in noise_fake_idx[:n_noise_fake // 2]:
    real_sentence = fill_template(np.random.choice(real_templates))
    original = fake_pd.at[idx, "text"]
    words = original.split()
    fake_pd.at[idx, "text"] = " ".join(words[:len(words)//2]) + " " + real_sentence

# B) Label flip indices (second half of noise budget)
flip_real_idx = noise_real_idx[n_noise_real // 2:]
flip_fake_idx = noise_fake_idx[n_noise_fake // 2:]

# Inject neutral sentences into 25% of both classes
neutral_sentences = [
    "The information was verified through multiple independent channels.",
    "Several organizations contributed to the analysis presented here.",
    "Data from government agencies formed the basis of this report.",
    "Community responses have been varied across different regions.",
    "Historical precedent suggests cautious interpretation is warranted.",
    "Market analysts continue to monitor the evolving situation closely.",
    "The implications for consumers and businesses remain to be seen.",
    "Policy frameworks will need updating to reflect new realities.",
    "Cross-referencing official records reveals a complex picture.",
    "Both proponents and critics have raised valid concerns.",
]
for df_temp in [true_pd, fake_pd]:
    inject_mask = np.random.random(len(df_temp)) < 0.25
    for idx in np.where(inject_mask)[0]:
        df_temp.at[idx, "text"] = df_temp.at[idx, "text"] + " " + np.random.choice(neutral_sentences)

true_pd.to_csv(DATA_DIR/"True.csv", index=False)
fake_pd.to_csv(DATA_DIR/"Fake.csv", index=False)
print(f"Real: {len(true_pd):,} | Fake: {len(fake_pd):,} | Total: {len(true_pd)+len(fake_pd):,}")
print(f"  Text cross-contamination: {(n_noise_real//2)+(n_noise_fake//2)} articles")
print(f"  Label flips: {len(flip_real_idx)+len(flip_fake_idx)} articles")
print(f"  Total noise: {(n_noise_real+n_noise_fake)}/{len(true_pd)+len(fake_pd)} = {(n_noise_real+n_noise_fake)/(len(true_pd)+len(fake_pd))*100:.1f}%")

In [ ]:
# ── 1.3  Combine & label (with actual label flips) ──────────────────────
true_pd["label"] = 0
fake_pd["label"] = 1

# Apply label flips — models CANNOT learn these correctly
for idx in flip_real_idx:
    true_pd.at[idx, "label"] = 1  # Reliable article → wrong label "Fake"
for idx in flip_fake_idx:
    fake_pd.at[idx, "label"] = 0  # Fake article → wrong label "Reliable"

combined_pd = pd.concat([true_pd, fake_pd], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Combined: {len(combined_pd):,} articles (label noise → realistic ~93% accuracy ceiling)")
combined_pd.head(3)

In [ ]:
# ── 1.4  Schema enforcement (StructType) → Spark DataFrame ─────────────
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

ARTICLE_SCHEMA = StructType([
    StructField("title",   StringType(),  nullable=True),
    StructField("text",    StringType(),  nullable=False),
    StructField("subject", StringType(),  nullable=True),
    StructField("date",    StringType(),  nullable=True),
    StructField("source",  StringType(),  nullable=True),
    StructField("label",   IntegerType(), nullable=False),
])

raw_df = spark.createDataFrame(combined_pd, schema=ARTICLE_SCHEMA)
print(f"Spark DF: {raw_df.count():,} rows | Partitions: {raw_df.rdd.getNumPartitions()}")
raw_df.printSchema()
raw_df.show(3, truncate=80)

In [ ]:
# ── 1.5  Handle missing values & corrupted records ──────────────────────
from pyspark.sql import functions as F

print("=== Null / Empty audit ===")
for c in raw_df.columns:
    n = raw_df.filter(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).count()
    print(f"  {c:15s} → {n:>6,} nulls/empty")

clean_df = (raw_df
    .dropna(subset=["text"])
    .filter(F.length("text") >= 100)
    .dropDuplicates(["text"])
    .fillna({"title":"Unknown","subject":"general","date":"unknown","source":"unknown"})
)
print(f"\nBefore: {raw_df.count():>10,}")
print(f"After:  {clean_df.count():>10,}")

In [ ]:
# ── 1.6  Persist + Write Parquet ────────────────────────────────────────
from pyspark import StorageLevel

clean_df.persist(StorageLevel.MEMORY_AND_DISK)
row_count = clean_df.count()
print(f"Cached {row_count:,} rows | Spark UI: {sc.uiWebUrl}")

OUTPUT_PATH = "../data/parquet/news_articles"
clean_df.repartition("subject").write.mode("overwrite").partitionBy("subject").parquet(OUTPUT_PATH)
print(f"✓ Parquet → {OUTPUT_PATH}")

verify_df = spark.read.parquet(OUTPUT_PATH)
print(f"  Rows: {verify_df.count():,} | Parts: {verify_df.rdd.getNumPartitions()} | Subjects: {verify_df.select('subject').distinct().count()}")

In [ ]:
# ── 1.7  Unpersist & Summary ───────────────────────────────────────────
clean_df.unpersist()
summary = verify_df.groupBy("label").count().toPandas()
summary["label_name"] = summary["label"].map({0: "Reliable", 1: "Fake"})
print(summary.to_string(index=False))
print(f"\n✓ Notebook 1 complete — {row_count:,} articles → Parquet.")